# Preview engineered features without loading the full datasets

Run the cell below in Colab. It gets dataset sizes from Parquet metadata and reads only five rows of a few predictor columns from each split. The returned `X_preview`, `y_preview`, and `keys_preview` are samples only; they are not training inputs. Change `CASE` to preview another feature variant. The full loader for model training is in [README.md](README.md).

In [1]:
import json
from pathlib import Path

import pandas as pd
import pyarrow.parquet as pq
from IPython.display import display
from google.colab import drive

drive.mount("/content/drive")
FEATURE_ROOT = Path(
    "/content/drive/MyDrive/Colab Notebooks/"
    "BT4222 Shared Final Folder/feature_engineering"
)
CASE_DIRS = {
    "raw": FEATURE_ROOT,
    "log": FEATURE_ROOT / "log_variants",
    "scaled": FEATURE_ROOT / "scaled_log_variants",
    "log_scaled": FEATURE_ROOT / "scaled_log_variants",
}
NON_PREDICTORS = {
    "split", "user_id", "impression_id", "candidate_article_id",
    "current_page_article_id", "impression_time", "clicked",
}
KEY_COLUMNS = ["user_id", "impression_id", "candidate_article_id"]


def preview_xy(split, case="raw", n=5):
    if split not in {"train", "validation", "test"}:
        raise ValueError(f"Unknown split: {split}")
    if case not in CASE_DIRS:
        raise ValueError(f"Unknown case: {case}")
    if n < 1:
        raise ValueError("n must be at least 1")

    path = CASE_DIRS[case] / f"{split}_features.parquet"
    if not path.is_file():
        raise FileNotFoundError(path)
    parquet = pq.ParquetFile(path)
    available = parquet.schema_arrow.names

    if case in {"log", "log_scaled"}:
        selection_path = FEATURE_ROOT / "log_variants/log_variant_selection.json"
        source_fields = json.loads(selection_path.read_text())["source_fields"]
        missing = {f"{field}_log1p" for field in source_fields} - set(available)
        if missing:
            raise ValueError(f"Log fields missing from {path}: {sorted(missing)}")
        excluded = set(source_fields)
    else:
        excluded = {field for field in available if field.endswith("_log1p")}

    feature_columns = [
        field for field in available
        if field not in NON_PREDICTORS and field not in excluded
    ]
    required = KEY_COLUMNS + (["clicked"] if split != "test" else [])
    missing = set(required) - set(available)
    if missing:
        raise ValueError(f"Columns missing from {path}: {sorted(missing)}")

    sample_features = feature_columns[:8]
    first_log = next((field for field in feature_columns if field.endswith("_log1p")), None)
    if first_log and first_log not in sample_features:
        sample_features.append(first_log)
    read_columns = required + sample_features
    first_batch = next(parquet.iter_batches(batch_size=n, columns=read_columns), None)
    sample = first_batch.to_pandas() if first_batch is not None else pd.DataFrame(columns=read_columns)

    X_preview = sample[sample_features]
    y_preview = sample["clicked"] if split != "test" else None
    keys_preview = sample[KEY_COLUMNS]
    print(f"{split}: {parquet.metadata.num_rows:,} rows; {len(feature_columns)} predictors; "
          f"{path.stat().st_size / 2**20:.1f} MiB on Drive")
    return X_preview, y_preview, keys_preview, feature_columns


CASE = "raw"  # "raw", "log", "scaled", or "log_scaled"
for split in ("train", "validation", "test"):
    X_preview, y_preview, keys_preview, feature_columns = preview_xy(split, CASE)
    parts = [keys_preview, X_preview]
    if y_preview is not None:
        parts.append(y_preview)
    display(pd.concat(parts, axis=1))

# To inspect all predictor names without reading more rows:
print("Predictor columns:", feature_columns)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
train: 2,585,747 rows; 202 predictors; 154.9 MiB on Drive


,user_id,impression_id,candidate_article_id,is_sso_user,is_subscriber,is_front_page,hour,day_of_week,slate_size,is_weekend,premium,clicked
0,139836,149474,9778623,False,False,True,7,2,6,False,False,0
1,139836,149474,9778682,False,False,True,7,2,6,False,False,0
2,139836,149474,9778669,False,False,True,7,2,6,False,False,0
3,139836,149474,9778657,False,False,True,7,2,6,False,False,1
4,139836,149474,9778736,False,False,True,7,2,6,False,False,0


validation: 344,666 rows; 202 predictors; 18.9 MiB on Drive


,user_id,impression_id,candidate_article_id,is_sso_user,is_subscriber,is_front_page,hour,day_of_week,slate_size,is_weekend,premium,clicked
0,1026339,221666,9784094,False,False,True,15,5,8,True,False,0
1,1026339,221666,9782449,False,False,True,15,5,8,True,False,0
2,1026339,221666,9784056,False,False,True,15,5,8,True,False,0
3,1026339,221666,9784214,False,False,True,15,5,8,True,False,0
4,1026339,221666,9780373,False,False,True,15,5,8,True,False,0


test: 446,915 rows; 202 predictors; 19.5 MiB on Drive


,user_id,impression_id,candidate_article_id,is_sso_user,is_subscriber,is_front_page,hour,day_of_week,slate_size,is_weekend,premium
0,44781,6453754,9787848,False,False,True,15,0,6,False,False
1,44781,6453754,9798724,False,False,True,15,0,6,False,False
2,44781,6453754,9793856,False,False,True,15,0,6,False,False
3,44781,6453754,9798498,False,False,True,15,0,6,False,False
4,44781,6453754,9798906,False,False,True,15,0,6,False,False


Predictor columns: ['is_sso_user', 'is_subscriber', 'is_front_page', 'hour', 'day_of_week', 'slate_size', 'is_weekend', 'premium', 'body_word_count', 'topic_count', 'has_topics', 'article_age_h', 'article_age_missing', 'is_24h_eligible', 'slate_relative_age_h', 'history_size', 'median_history_body_word_count', 'median_history_read_time', 'mean_history_scroll_percentage', 'history_body_word_count_missing', 'history_read_time_missing', 'history_scroll_missing', 'history_clicks_7d', 'candidate_category_preference', 'candidate_category_preference_missing', 'candidate_article_type_preference', 'candidate_article_type_preference_missing', 'candidate_topic_preference', 'candidate_topic_preference_missing', 'category_affinity_72h', 'category_affinity_missing', 'last5_mean_topic_jaccard', 'topic_affinity_missing', 'has_any_topic_overlap', 'same_category_mean_read_time', 'same_category_read_depth_missing', 'hist_category_frac_000', 'hist_category_frac_001', 'hist_category_frac_002', 'hist_catego

In [2]:
import json
from pathlib import Path

import pandas as pd
import pyarrow.parquet as pq
from IPython.display import display
from google.colab import drive

drive.mount("/content/drive")
FEATURE_ROOT = Path(
    "/content/drive/MyDrive/Colab Notebooks/"
    "BT4222 Shared Final Folder/feature_engineering"
)
CASE_DIRS = {
    "raw": FEATURE_ROOT,
    "log": FEATURE_ROOT / "log_variants",
    "scaled": FEATURE_ROOT / "scaled_log_variants",
    "log_scaled": FEATURE_ROOT / "scaled_log_variants",
}
NON_PREDICTORS = {
    "split", "user_id", "impression_id", "candidate_article_id",
    "current_page_article_id", "impression_time", "clicked",
}
KEY_COLUMNS = ["user_id", "impression_id", "candidate_article_id"]


def preview_xy(split, case="raw", n=5):
    if split not in {"train", "validation", "test"}:
        raise ValueError(f"Unknown split: {split}")
    if case not in CASE_DIRS:
        raise ValueError(f"Unknown case: {case}")
    if n < 1:
        raise ValueError("n must be at least 1")

    path = CASE_DIRS[case] / f"{split}_features.parquet"
    if not path.is_file():
        raise FileNotFoundError(path)
    parquet = pq.ParquetFile(path)
    available = parquet.schema_arrow.names

    if case in {"log", "log_scaled"}:
        selection_path = FEATURE_ROOT / "log_variants/log_variant_selection.json"
        source_fields = json.loads(selection_path.read_text())["source_fields"]
        missing = {f"{field}_log1p" for field in source_fields} - set(available)
        if missing:
            raise ValueError(f"Log fields missing from {path}: {sorted(missing)}")
        excluded = set(source_fields)
    else:
        excluded = {field for field in available if field.endswith("_log1p")}

    feature_columns = [
        field for field in available
        if field not in NON_PREDICTORS and field not in excluded
    ]
    required = KEY_COLUMNS + (["clicked"] if split != "test" else [])
    missing = set(required) - set(available)
    if missing:
        raise ValueError(f"Columns missing from {path}: {sorted(missing)}")

    sample_features = feature_columns[:8]
    first_log = next((field for field in feature_columns if field.endswith("_log1p")), None)
    if first_log and first_log not in sample_features:
        sample_features.append(first_log)
    read_columns = required + sample_features
    first_batch = next(parquet.iter_batches(batch_size=n, columns=read_columns), None)
    sample = first_batch.to_pandas() if first_batch is not None else pd.DataFrame(columns=read_columns)

    X_preview = sample[sample_features]
    y_preview = sample["clicked"] if split != "test" else None
    keys_preview = sample[KEY_COLUMNS]
    print(f"{split}: {parquet.metadata.num_rows:,} rows; {len(feature_columns)} predictors; "
          f"{path.stat().st_size / 2**20:.1f} MiB on Drive")
    return X_preview, y_preview, keys_preview, feature_columns


CASE = "log"  # "raw", "log", "scaled", or "log_scaled"
for split in ("train", "validation", "test"):
    X_preview, y_preview, keys_preview, feature_columns = preview_xy(split, CASE)
    parts = [keys_preview, X_preview]
    if y_preview is not None:
        parts.append(y_preview)
    display(pd.concat(parts, axis=1))

# To inspect all predictor names without reading more rows:
print("Predictor columns:", feature_columns)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
train: 2,585,747 rows; 202 predictors; 182.4 MiB on Drive


,user_id,impression_id,candidate_article_id,is_sso_user,is_subscriber,is_front_page,hour,day_of_week,is_weekend,premium,topic_count,median_history_body_word_count_log1p,clicked
0,139836,149474,9778623,False,False,True,7,2,False,False,3,5.713733,0
1,139836,149474,9778682,False,False,True,7,2,False,False,3,5.713733,0
2,139836,149474,9778669,False,False,True,7,2,False,False,4,5.713733,0
3,139836,149474,9778657,False,False,True,7,2,False,False,3,5.713733,1
4,139836,149474,9778736,False,False,True,7,2,False,False,7,5.713733,0


validation: 344,666 rows; 202 predictors; 22.7 MiB on Drive


,user_id,impression_id,candidate_article_id,is_sso_user,is_subscriber,is_front_page,hour,day_of_week,is_weekend,premium,topic_count,median_history_body_word_count_log1p,clicked
0,1026339,221666,9784094,False,False,True,15,5,True,False,4,5.57973,0
1,1026339,221666,9782449,False,False,True,15,5,True,False,3,5.57973,0
2,1026339,221666,9784056,False,False,True,15,5,True,False,4,5.57973,0
3,1026339,221666,9784214,False,False,True,15,5,True,False,5,5.57973,0
4,1026339,221666,9780373,False,False,True,15,5,True,False,1,5.57973,0


test: 446,915 rows; 202 predictors; 24.2 MiB on Drive


,user_id,impression_id,candidate_article_id,is_sso_user,is_subscriber,is_front_page,hour,day_of_week,is_weekend,premium,topic_count,median_history_body_word_count_log1p
0,44781,6453754,9787848,False,False,True,15,0,False,False,2,5.616771
1,44781,6453754,9798724,False,False,True,15,0,False,False,5,5.616771
2,44781,6453754,9793856,False,False,True,15,0,False,False,3,5.616771
3,44781,6453754,9798498,False,False,True,15,0,False,False,3,5.616771
4,44781,6453754,9798906,False,False,True,15,0,False,False,3,5.616771


Predictor columns: ['is_sso_user', 'is_subscriber', 'is_front_page', 'hour', 'day_of_week', 'is_weekend', 'premium', 'topic_count', 'has_topics', 'article_age_missing', 'is_24h_eligible', 'slate_relative_age_h', 'history_size', 'mean_history_scroll_percentage', 'history_body_word_count_missing', 'history_read_time_missing', 'history_scroll_missing', 'candidate_category_preference', 'candidate_category_preference_missing', 'candidate_article_type_preference', 'candidate_article_type_preference_missing', 'candidate_topic_preference', 'candidate_topic_preference_missing', 'category_affinity_72h', 'category_affinity_missing', 'last5_mean_topic_jaccard', 'topic_affinity_missing', 'has_any_topic_overlap', 'same_category_read_depth_missing', 'hist_category_frac_000', 'hist_category_frac_001', 'hist_category_frac_002', 'hist_category_frac_003', 'hist_category_frac_004', 'hist_category_frac_005', 'hist_category_frac_006', 'hist_category_frac_007', 'hist_category_frac_008', 'hist_category_frac_0